# 7.7 奖励黑客与过度优化深挖

> 🕐 预估学习时间：40分钟

当优化的是奖励模型而非真偏好时，策略会钻空子：空话加长、谄媚、格式刷分、不安全但高分。本节用可控玩具复现 Goodhart 效应与缓解手段。

深挖点：
- 奖励模型盲区 → 策略利用
- KL 约束与早期停止
- 长度偏见
- 集成奖励 / 对抗挖掘


## 1. Goodhart：代理指标被刷爆

真偏好看“正确性”，奖励模型误把“长且自信”当好。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)


def true_pref(features):
    # features: [correctness, length, confidence]
    return 2.0 * features[:, 0] - 0.1 * features[:, 1]


def proxy_rm(features):
    # mistakenly loves length + confidence
    return 0.5 * features[:, 0] + 0.8 * features[:, 1] + 0.5 * features[:, 2]


# policy samples features via parameters
class Policy(nn.Module):
    def __init__(self):
        super().__init__()
        self.logit = nn.Parameter(torch.zeros(3))  # bernoulli logits for traits

    def sample(self, n=64):
        p = torch.sigmoid(self.logit)
        feats = torch.bernoulli(p.expand(n, -1))
        # length continuous-ish
        feats = feats.clone()
        feats[:, 1] = torch.sigmoid(self.logit[1]) * (1 + 0.1 * torch.randn(n))
        return feats, p


pol = Policy()
opt = torch.optim.Adam(pol.parameters(), lr=0.2)
print('=== Optimize Proxy RM ===')
for step in range(40):
    feats, p = pol.sample()
    reward = proxy_rm(feats).mean()
    # maximize reward
    loss = -reward
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 10 == 0 or step == 39:
        with torch.no_grad():
            f, _ = pol.sample(200)
            print(f'step={step} proxy={proxy_rm(f).mean():.3f} true={true_pref(f).mean():.3f} '
                  f'p={torch.sigmoid(pol.logit).tolist()}')
print('Key: Proxy goes up while true preference collapses — classic reward hacking.')


## 2. KL / 距离约束缓解

相对参考策略加 KL 惩罚，限制跑出分布太远。


In [ ]:
ref_logit = torch.zeros(3)
pol2 = Policy()
opt2 = torch.optim.Adam(pol2.parameters(), lr=0.2)
print('=== Proxy + KL to reference ===')
for step in range(40):
    feats, p = pol2.sample()
    reward = proxy_rm(feats).mean()
    pref = torch.sigmoid(pol2.logit)
    pref_ref = torch.sigmoid(ref_logit)
    kl = (pref * (pref.clamp_min(1e-6).log() - pref_ref.clamp_min(1e-6).log())
          + (1 - pref) * ((1 - pref).clamp_min(1e-6).log() - (1 - pref_ref).clamp_min(1e-6).log())).sum()
    loss = -reward + 0.5 * kl
    opt2.zero_grad(); loss.backward(); opt2.step()
    if step % 10 == 0 or step == 39:
        with torch.no_grad():
            f, _ = pol2.sample(200)
            print(f'step={step} proxy={proxy_rm(f).mean():.3f} true={true_pref(f).mean():.3f} kl={kl.item():.3f}')
print('Key: KL buys safety margin but does not fix a systematically wrong RM.')


## 3. 长度偏见校正

奖励减去长度基线或使用长度归一化（SimPO 风格直觉）。


In [ ]:
def length_normalized_rm(features):
    return proxy_rm(features) - 0.8 * features[:, 1]


pol3 = Policy()
opt3 = torch.optim.Adam(pol3.parameters(), lr=0.2)
print('=== Length-normalized proxy ===')
for step in range(40):
    feats, _ = pol3.sample()
    loss = -length_normalized_rm(feats).mean()
    opt3.zero_grad(); loss.backward(); opt3.step()
    if step % 10 == 0 or step == 39:
        f, _ = pol3.sample(200)
        print(f'step={step} true={true_pref(f).mean():.3f} mean_len={f[:,1].mean():.3f}')
print('Key: Explicitly debias known RM failure modes before policy optimization.')


## 4. 对抗挖掘与集成奖励

定期用攻击提示搜索高 RM、低真偏好样本，加入 RM 再训练；或多 RM 取 min。


In [ ]:
def ensemble_min_rm(features):
    rm2 = 1.5 * features[:, 0] - 0.2 * features[:, 2]  # another head
    return torch.minimum(proxy_rm(features), rm2)


# mine hacks: high proxy, low true
cand = torch.rand(1000, 3)
hack = cand[(proxy_rm(cand) > 1.2) & (true_pref(cand) < 0)]
print('=== Adversarial Mining ===')
print(f'mined hacks={len(hack)} / 1000')
if len(hack):
    print('example', hack[0].tolist(), 'proxy', proxy_rm(hack[:1]).item(), 'true', true_pref(hack[:1]).item())
print(f'ensemble example mean={ensemble_min_rm(cand[:8]).mean():.3f}')
print('Key: Close the loop—mine hacks, retrain RM, constrain policy, repeat.')


## 课后思考题

1. 如何用影子真偏好（人工抽检）校准线上 RM？
2. GRPO/过程奖励是否引入新的可黑客点？
3. 长度归一化会不会伤害真正需要长推理的任务？
4. 发现黑客后，该优先修 RM 还是加规则护栏？

---
> 本节是奖励黑客与过度优化的垂直深挖。建议对照真实训练日志/线上指标复现关键实验，而不是只跑通玩具代码。
